# PFT Perturbation: control vs. perturbed CTSM run

This notebook:
1. Runs `run_neon_v2.py --setup-only` once to download the baseline NEON surface dataset.
2. Loads `surfdata_*.nc` and **shifts a percentage of cover from one PFT to another** (keeping the column sum at 100%).
3. Saves the perturbed surface dataset.
4. Launches **two CTSM simulations** — a control (default `fsurdat`) and a perturbed (`fsurdat` repointed at the modified file).
5. Loads the resulting CLM history files and compares key ecosystem variables (TLAI, GPP, TOTECOSYSC).

**Runtime warning** — a single transient NEON simulation typically takes **1–4+ hours** on this container. Cells 1–6 are fast (seconds). Cells 7–8 are the slow ones — run them when you're ready to wait.

## 1. Configuration & imports

In [ ]:
import os, sys, glob, shutil, subprocess, time
from pathlib import Path

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# ---------------- EDIT ME ----------------
SITE              = "ABBY"          # 4-letter NEON site code
SOURCE_PFT_IDX    = 1               # PFT to take cover FROM (see PFT table below)
TARGET_PFT_IDX    = 13              # PFT to give cover TO
SHIFT_PERCENT     = 20.0            # absolute percentage points to move
EXPERIMENT_TAG    = f"shift{SOURCE_PFT_IDX}to{TARGET_PFT_IDX}_{int(SHIFT_PERCENT)}pct"

WORK_ROOT         = Path("/home/user/CLM-NEON-pft")  # all output goes here
CONTROL_ROOT      = WORK_ROOT / "control"
PERTURBED_ROOT    = WORK_ROOT / f"perturbed_{EXPERIMENT_TAG}"
PERTURBED_SURFDAT = WORK_ROOT / f"surfdata_{SITE}_{EXPERIMENT_TAG}.nc"
# -----------------------------------------

RUN_NEON_V2 = "/opt/ncar/cesm/tools/site_and_regional/run_neon_v2.py"

WORK_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Site            : {SITE}")
print(f"Experiment      : {EXPERIMENT_TAG}")
print(f"Control root    : {CONTROL_ROOT}")
print(f"Perturbed root  : {PERTURBED_ROOT}")
print(f"Perturbed surfd.: {PERTURBED_SURFDAT}")

## 2. CLM-PFT reference table

Standard CLM5 17-PFT indexing. Use these indices for `SOURCE_PFT_IDX` and `TARGET_PFT_IDX` above.

In [ ]:
PFT_NAMES = {
    0:  "not_vegetated",
    1:  "needleleaf_evergreen_temperate_tree",
    2:  "needleleaf_evergreen_boreal_tree",
    3:  "needleleaf_deciduous_boreal_tree",
    4:  "broadleaf_evergreen_tropical_tree",
    5:  "broadleaf_evergreen_temperate_tree",
    6:  "broadleaf_deciduous_tropical_tree",
    7:  "broadleaf_deciduous_temperate_tree",
    8:  "broadleaf_deciduous_boreal_tree",
    9:  "broadleaf_evergreen_shrub",
    10: "broadleaf_deciduous_temperate_shrub",
    11: "broadleaf_deciduous_boreal_shrub",
    12: "c3_arctic_grass",
    13: "c3_non_arctic_grass",
    14: "c4_grass",
    15: "unmanaged_crop",
    16: "managed_crop",
}
for idx, name in PFT_NAMES.items():
    print(f"  {idx:2d}  {name}")

## 3. Warm-up setup-only run — downloads baseline NEON surfdata

This is fast (no simulation, just network + namelist generation). It creates the control case and triggers CTSM's `check_all_input_data` step, which downloads the NEON surface dataset to `DIN_LOC_ROOT`.

In [ ]:
def run(cmd, cwd=None):
    """Helper that streams subprocess output to the notebook."""
    print(f"$ {' '.join(map(str, cmd))}")
    return subprocess.run(cmd, cwd=cwd, check=True)

if not (CONTROL_ROOT / f"{SITE}.transient").exists():
    run([
        RUN_NEON_V2,
        "--neon-sites", SITE,
        "--output-root", str(CONTROL_ROOT),
        "--setup-only", "--overwrite",
    ])
else:
    print(f"Control case already exists at {CONTROL_ROOT / (SITE + '.transient')}, skipping warm-up.")

## 4. Locate and inspect the baseline surface dataset

In [ ]:
def locate_baseline_surfdata(case_dir: Path, site: str) -> Path:
    """Inspect the case's lnd_in namelist to find the active fsurdat path."""
    rundir = subprocess.check_output(
        ["./xmlquery", "-value", "RUNDIR"], cwd=case_dir
    ).decode().strip()
    lnd_in = Path(rundir) / "lnd_in"
    if not lnd_in.exists():
        raise FileNotFoundError(f"{lnd_in} not generated yet; rerun setup.")
    for line in lnd_in.read_text().splitlines():
        line = line.strip()
        if line.startswith("fsurdat"):
            return Path(line.split("=", 1)[1].strip().strip("'\""))
    raise RuntimeError("fsurdat not found in lnd_in.")

case_dir = CONTROL_ROOT / f"{SITE}.transient"
baseline_surfdat = locate_baseline_surfdata(case_dir, SITE)
print(f"Baseline surfdata: {baseline_surfdat}")

ds_base = xr.open_dataset(baseline_surfdat)
print("\nVariables of interest:")
print("  PCT_NAT_PFT  shape =", ds_base["PCT_NAT_PFT"].shape, "(dims:", ds_base["PCT_NAT_PFT"].dims, ")")
print("  PCT_NATVEG   shape =", ds_base["PCT_NATVEG"].shape)

pct_pft = ds_base["PCT_NAT_PFT"].values.squeeze()  # shape (17,) for single-point
print("\nCurrent PCT_NAT_PFT distribution:")
for idx, pct in enumerate(pct_pft):
    if pct > 0.01:
        print(f"  PFT {idx:2d} ({PFT_NAMES[idx]:42s})  {pct:6.2f}%")
print(f"  sum = {pct_pft.sum():.2f}%")

## 5. Apply the PFT shift

Moves `SHIFT_PERCENT` percentage points from `SOURCE_PFT_IDX` to `TARGET_PFT_IDX`. Validates that the source has enough cover and the column sum stays at 100%.

In [ ]:
def shift_pft(pct_pft_arr: np.ndarray, src: int, dst: int, amount: float) -> np.ndarray:
    """Return a copy of pct_pft_arr with `amount` percentage points shifted src -> dst."""
    out = pct_pft_arr.copy()
    if out[src] < amount:
        raise ValueError(
            f"Cannot shift {amount}% from PFT {src} ({PFT_NAMES[src]}): only {out[src]:.2f}% available."
        )
    out[src] -= amount
    out[dst] += amount
    if not np.isclose(out.sum(), pct_pft_arr.sum(), atol=1e-6):
        raise RuntimeError(f"Sum changed: {pct_pft_arr.sum()} -> {out.sum()}")
    return out

pct_pft_perturbed = shift_pft(pct_pft, SOURCE_PFT_IDX, TARGET_PFT_IDX, SHIFT_PERCENT)

# Side-by-side comparison plot
fig, ax = plt.subplots(figsize=(12, 4))
x = np.arange(len(pct_pft))
w = 0.4
ax.bar(x - w/2, pct_pft, w, label="baseline", color="#4477aa")
ax.bar(x + w/2, pct_pft_perturbed, w, label=f"perturbed (shift {SHIFT_PERCENT}% PFT{SOURCE_PFT_IDX}→{TARGET_PFT_IDX})", color="#ee6677")
ax.set_xticks(x)
ax.set_xticklabels([str(i) for i in x])
ax.set_xlabel("PFT index")
ax.set_ylabel("% cover")
ax.set_title(f"{SITE} PCT_NAT_PFT: baseline vs perturbed")
ax.legend()
plt.tight_layout()
plt.show()

print("Changes:")
print(f"  PFT {SOURCE_PFT_IDX} ({PFT_NAMES[SOURCE_PFT_IDX]}): {pct_pft[SOURCE_PFT_IDX]:.2f}% -> {pct_pft_perturbed[SOURCE_PFT_IDX]:.2f}%")
print(f"  PFT {TARGET_PFT_IDX} ({PFT_NAMES[TARGET_PFT_IDX]}): {pct_pft[TARGET_PFT_IDX]:.2f}% -> {pct_pft_perturbed[TARGET_PFT_IDX]:.2f}%")

## 6. Save the perturbed surface dataset

Copies the original NetCDF and overwrites `PCT_NAT_PFT` in place. All other variables (soil texture, topography, etc.) are preserved.

In [ ]:
shutil.copy2(baseline_surfdat, PERTURBED_SURFDAT)

with xr.open_dataset(PERTURBED_SURFDAT) as _ds:
    ds_perturbed = _ds.load()

arr = ds_perturbed["PCT_NAT_PFT"]
if arr.ndim == 3:        # (natpft, lsmlat, lsmlon)
    arr.values[:, 0, 0] = pct_pft_perturbed
elif arr.ndim == 1:      # (natpft,)
    arr.values[:] = pct_pft_perturbed
else:
    raise RuntimeError(f"Unexpected PCT_NAT_PFT rank: {arr.ndim}")

ds_perturbed.to_netcdf(PERTURBED_SURFDAT, mode="w")
ds_perturbed.close()

# Re-read to verify
with xr.open_dataset(PERTURBED_SURFDAT) as verify:
    saved = verify["PCT_NAT_PFT"].values.squeeze()
    print("Saved perturbed surfdata: ", PERTURBED_SURFDAT)
    print("Verification (saved vs. expected):")
    print("  PFT", SOURCE_PFT_IDX, ":", saved[SOURCE_PFT_IDX], "vs", pct_pft_perturbed[SOURCE_PFT_IDX])
    print("  PFT", TARGET_PFT_IDX, ":", saved[TARGET_PFT_IDX], "vs", pct_pft_perturbed[TARGET_PFT_IDX])

## 7. Run CONTROL simulation (slow — hours)

This calls `run_neon_v2.py` again without `--setup-only` and with `--no-batch`, so it builds (if needed) and runs synchronously.

In [ ]:
def submit_case(case_dir: Path):
    """Build (if needed) and submit a CTSM case synchronously with no batch."""
    if not (case_dir / "CaseStatus").exists():
        raise RuntimeError(f"Not a built case dir: {case_dir}")
    run(["./case.build"], cwd=case_dir)
    run(["./case.submit", "--no-batch"], cwd=case_dir)
    return case_dir

control_case = CONTROL_ROOT / f"{SITE}.transient"
print("Running CONTROL case... grab a coffee.\n")
t0 = time.time()
submit_case(control_case)
print(f"\nControl finished in {(time.time()-t0)/60:.1f} min")

## 8. Run PERTURBED simulation (slow — hours)

Setup-only first (so we can edit `user_nl_clm` to point at the perturbed surfdata), then build + submit.

In [ ]:
if not (PERTURBED_ROOT / f"{SITE}.transient").exists():
    run([
        RUN_NEON_V2,
        "--neon-sites", SITE,
        "--output-root", str(PERTURBED_ROOT),
        "--setup-only", "--overwrite",
    ])

# Override fsurdat in the perturbed case
perturbed_case = PERTURBED_ROOT / f"{SITE}.transient"
user_nl = perturbed_case / "user_nl_clm"
current = user_nl.read_text()
if "fsurdat" not in current:
    with user_nl.open("a") as fd:
        fd.write(f"\nfsurdat = '{PERTURBED_SURFDAT}'\n")
    print(f"Appended fsurdat -> {PERTURBED_SURFDAT}")
else:
    print("fsurdat already present in user_nl_clm; not modifying. Inspect manually if unsure.")

# Force CTSM to re-resolve namelists with the new fsurdat
run(["./preview_namelists"], cwd=perturbed_case)

print("\nRunning PERTURBED case... grab another coffee.\n")
t0 = time.time()
submit_case(perturbed_case)
print(f"\nPerturbed finished in {(time.time()-t0)/60:.1f} min")

## 9. Load history files from both runs

In [ ]:
def load_hist(case_dir: Path, site: str) -> xr.Dataset:
    """Locate CLM h1 history files for the case and open them as a single dataset."""
    rundir = subprocess.check_output(
        ["./xmlquery", "-value", "RUNDIR"], cwd=case_dir
    ).decode().strip()
    dout = subprocess.check_output(
        ["./xmlquery", "-value", "DOUT_S_ROOT"], cwd=case_dir
    ).decode().strip()
    pattern_run  = f"{rundir}/{site}.transient.clm2.h1.*.nc"
    pattern_dout = f"{dout}/lnd/hist/{site}.transient.clm2.h1.*.nc"
    files = sorted(glob.glob(pattern_dout) or glob.glob(pattern_run))
    if not files:
        raise FileNotFoundError(f"No h1 files for {case_dir}; checked {pattern_dout} and {pattern_run}")
    print(f"{case_dir.name}: {len(files)} h1 file(s)")
    return xr.open_mfdataset(files, combine="by_coords", decode_times=True)

ds_control   = load_hist(control_case,   SITE)
ds_perturbed = load_hist(perturbed_case, SITE)
print("\nVariables available:", list(ds_control.data_vars)[:15], "...")

## 10. Time-series comparison — control vs perturbed

In [ ]:
VARS_TO_COMPARE = ["TLAI", "GPP", "TOTECOSYSC", "TOTVEGC", "H2OSNO"]

fig, axes = plt.subplots(len(VARS_TO_COMPARE), 2, figsize=(14, 3 * len(VARS_TO_COMPARE)))
for i, var in enumerate(VARS_TO_COMPARE):
    if var not in ds_control or var not in ds_perturbed:
        axes[i, 0].set_title(f"{var} not found in outputs")
        continue
    c = ds_control[var].squeeze()
    p = ds_perturbed[var].squeeze()
    diff = (p - c)

    axes[i, 0].plot(c.time, c.values, label="control", color="#4477aa")
    axes[i, 0].plot(p.time, p.values, label="perturbed", color="#ee6677")
    axes[i, 0].set_ylabel(f"{var} [{c.attrs.get('units','')}]")
    axes[i, 0].legend(loc="best")
    axes[i, 0].set_title(f"{var} time series")

    axes[i, 1].plot(diff.time, diff.values, color="k")
    axes[i, 1].axhline(0, color="grey", lw=0.5)
    axes[i, 1].set_ylabel(f"Δ{var}")
    axes[i, 1].set_title(f"perturbed − control")

axes[-1, 0].set_xlabel("time")
axes[-1, 1].set_xlabel("time")
plt.tight_layout()
plt.show()

## 11. Statistical summary using `analytics_modules.model_misfit`

Treats the control run as "observation" and the perturbed run as "prediction" to quantify how much the PFT shift moved each variable.

In [ ]:
from analytics_modules.model_misfit import residuals_plots

results = {}
for var in VARS_TO_COMPARE:
    if var not in ds_control or var not in ds_perturbed:
        continue
    c = ds_control[var].squeeze().values
    p = ds_perturbed[var].squeeze().values
    fig, residuals, metrics, conclusion = residuals_plots(c, p, bins=40, savepath=None)
    fig.suptitle(f"{var}: control (obs) vs perturbed (pred)", y=1.02)
    plt.show()
    print(f"{var:12s}  {conclusion}")
    results[var] = metrics

import pandas as pd
print("\nSummary metrics:")
print(pd.DataFrame(results).T)

## 12. Next steps

- Re-run cells 1–6 with different `SOURCE_PFT_IDX` / `TARGET_PFT_IDX` / `SHIFT_PERCENT` to build an ensemble.
- Use `analytics_modules.kalman_filter.kalman_gain_bias` to fit a calibration mapping from one experiment back to the control — the larger the required gain, the more sensitive the variable is to that PFT shift.
- If you want this to scale to many experiments, hoist the perturbation + launch logic into `analytics_modules/pft_perturbation.py` and call it from a loop.